# Extension 03 Minimal and pneumatic surfaces

In [ ]:
%pip install -q numpy session-py==0.310.0 session-jupyter==0.2.0

# Question

**How do iterative cotangent weights and follower normal pressures turn a linear Force Density net into a minimal soap film or an inflated pneumatic cushion?**

Inspired by the working solvers in `jikempff/Mite` (`FormFinding/MinimalSurface.cs` and `Geometry/DiscreteOperators.cs`) and `arpastrana/jax_fdm` (`equilibrium/loads.py` with `is_load_local=True` and `models.py` `equilibrium_iterative_xyz`).

**Given:** for zero load ${\color{#2e9e4a} p = 0}$, `# equation 6.32` is the stationary point of the weighted Dirichlet energy:

$$
\Phi(x, y, z) = \frac{1}{2}\left(u^{T}\,Q\,u + v^{T}\,Q\,v + w^{T}\,Q\,w\right) = \frac{1}{2}\sum_{k=0}^{m-1} q_k\,l_k^{2}
$$

- **Minimal surface (`Mite` `CotangentEdgeWeights`):** uniform $q_k = 1$ minimises squared edge lengths rather than surface area $\mathcal{A} = \sum_t A_t$. Freezing the discrete Laplace Beltrami cotangent weights at each outer step,

$$
q_k = \max\!\left(\tfrac{1}{2}\left(\cot\alpha_k + \cot\beta_k\right),\; q_{\min}\right), \qquad \cot\theta = \frac{a \cdot b}{\|a \times b\|}
$$

makes $\frac{1}{2}\sum_k q_k\,l_k^{2}$ equal the surface area at the current geometry, converging to a minimal surface in 15 linear solves!
- **Pneumatic follower pressure (`jax_fdm` `is_load_local=True`):** internal gauge pressure $\pi$ acts normal to the **deformed** triangles, contributing nodal loads ${\color{#2e9e4a} p_i(x)} = \sum_{t \ni i} \frac{1}{3}\,\pi\,\vec{s}_t$ where $\vec{s}_t = \frac{1}{2}(x_j - x_i)\times(x_k - x_i)$.

<div align="center">
  <img src="https://raw.githubusercontent.com/BRG-teaching/Teaching/main/CAS/logos/fd_equilibrium.svg"
       style="width:100%; height:auto;"
       alt="Equilibrium of membrane forces and normal surface loads at a free node" />
  <p style="text-align:center;">
    <em>Updating q from opposite triangle cotangents minimises surface area, while updating p along triangle normals inflates a pneumatic cushion.</em>
  </p>
</div>

- Compare a uniform $q$ net against `Mite`'s cotangent minimal surface on a warped saddle boundary
- Form find a minimal catenoid soap film between two coaxial rings and check Euler's analytical neck radius
- Inflate a square ETFE pillow under follower normal pressure (`jax_fdm`) and render the rotated normal loads

<hr style="border: none; border-top: 2px solid #555; margin: 2em 0;">

# Uniform Dirichlet Net Versus Cotangent Minimal Surface

Following `Mite` (`MinimalSurface.cs` and `DiscreteOperators.Cot`), we triangulate an $8 \times 8$ patch ($9 \times 9 = 81$ vertices, $128$ triangles) over a twisted saddle boundary where two opposite corners rise to $z = 2.6\text{ m}$. Notice how `Mite` computes $\cot\theta = (a \cdot b)/\|a \times b\|$ directly without trigonometric calls and clamps obtuse negative weights to $10^{-4}$ so $D_N = C_N^{T}\,Q\,C_N$ stays strictly positive definite.

In [1]:
import numpy as np
from session_py import Arrowhead, Color, Line, Matrix, Session
from session_jupyter import show

RED = Color(0.84, 0.16, 0.16, 1.0)
BLUE = Color(0.16, 0.44, 0.86, 1.0)
GREEN = Color(0.18, 0.62, 0.29, 1.0)
GRAY = Color(0.74, 0.74, 0.74, 1.0)


def build_tri_patch(n_side=9, span=6.0):
    """Build triangulated grid patch with alternating diagonals for symmetry."""
    gx, gy = np.meshgrid(np.linspace(0.0, span, n_side), np.linspace(0.0, span, n_side))
    x, y = gx.flatten(), gy.flatten()
    tris = []
    for j in range(n_side - 1):
        for i in range(n_side - 1):
            v00 = j * n_side + i
            v10 = v00 + 1
            v01 = v00 + n_side
            v11 = v01 + 1
            if (i + j) % 2 == 0:
                tris.append((v00, v10, v11))
                tris.append((v00, v11, v01))
            else:
                tris.append((v00, v10, v01))
                tris.append((v10, v11, v01))
    edge_set = set()
    for a, b, c in tris:
        for u, v in ((a, b), (b, c), (c, a)):
            edge_set.add((min(u, v), max(u, v)))
    edges = sorted(edge_set)
    edge_idx = {e: k for k, e in enumerate(edges)}
    C = np.zeros((len(edges), len(x)))
    for k, (a, b) in enumerate(edges):
        C[k, a], C[k, b] = 1.0, -1.0
    return x, y, tris, edges, edge_idx, C


def mesh_area(xyz, tris):
    p0 = xyz[[t[0] for t in tris]]
    p1 = xyz[[t[1] for t in tris]]
    p2 = xyz[[t[2] for t in tris]]
    return 0.5 * np.sum(np.linalg.norm(np.cross(p1 - p0, p2 - p0), axis=1))


def cotangent_weights(xyz, tris, edge_idx, num_edges, q_floor=1e-4):
    """Symmetric cotangent weights w_e = 0.5 * (cot alpha + cot beta) matching Mite."""
    w = np.zeros(num_edges)
    for i0, i1, i2 in tris:
        tri = (i0, i1, i2)
        for m in range(3):
            a, b, opp = tri[m], tri[(m + 1) % 3], tri[(m + 2) % 3]
            ua = xyz[a] - xyz[opp]
            ub = xyz[b] - xyz[opp]
            cross_norm = np.linalg.norm(np.cross(ua, ub))
            if cross_norm > 1e-10:
                cot_val = np.dot(ua, ub) / cross_norm
                e = edge_idx[(min(a, b), max(a, b))]
                w[e] += 0.5 * cot_val
    return np.maximum(w, q_floor)


n_s = 9
xs, ys, tris_s, edges_s, eidx_s, C_s = build_tri_patch(n_s, 6.0)
# prescribed boundary frame: harmonic saddle z = 2.6 * (x/6)*(1 - y/6) + ... -> high corners at (0,0) and (6,6)
zs = 2.6 * ((xs / 3.0 - 1.0) * (ys / 3.0 - 1.0) + 1.0) * 0.5
fixed_s = [k for k in range(len(xs)) if xs[k] in (0.0, 6.0) or ys[k] in (0.0, 6.0)]
free_s = [k for k in range(len(xs)) if k not in fixed_s]
C_Ns, C_Fs = C_s[:, free_s], C_s[:, fixed_s]

# 1. uniform q = 1.0 Dirichlet solve
xyz_uni = np.column_stack([xs.copy(), ys.copy(), zs.copy()])
Q_uni = np.eye(len(edges_s))

# equation 6.32
A_uni = C_Ns.T @ Q_uni @ C_Ns
xyz_uni[free_s, 0] = np.linalg.solve(A_uni, -C_Ns.T @ Q_uni @ C_Fs @ xyz_uni[fixed_s, 0])
xyz_uni[free_s, 1] = np.linalg.solve(A_uni, -C_Ns.T @ Q_uni @ C_Fs @ xyz_uni[fixed_s, 1])
xyz_uni[free_s, 2] = np.linalg.solve(A_uni, -C_Ns.T @ Q_uni @ C_Fs @ xyz_uni[fixed_s, 2])

# 2. Mite iterative cotangent minimal surface solve (15 outer updates)
xyz_min = xyz_uni.copy()
for step in range(15):
    q_cot = cotangent_weights(xyz_min, tris_s, eidx_s, len(edges_s))
    Q_cot = np.diag(q_cot)
    A_cot = C_Ns.T @ Q_cot @ C_Ns
    next_free = np.column_stack([
        np.linalg.solve(A_cot, -C_Ns.T @ Q_cot @ C_Fs @ xyz_min[fixed_s, 0]),
        np.linalg.solve(A_cot, -C_Ns.T @ Q_cot @ C_Fs @ xyz_min[fixed_s, 1]),
        np.linalg.solve(A_cot, -C_Ns.T @ Q_cot @ C_Fs @ xyz_min[fixed_s, 2]),
    ])
    disp = np.linalg.norm(next_free - xyz_min[free_s], axis=1).max()
    xyz_min[free_s] = next_free
    if disp < 1e-8:
        break

print("uniform q = 1 surface area      =", round(mesh_area(xyz_uni, tris_s), 4), "m^2")
print("cotangent minimal surface area  =", round(mesh_area(xyz_min, tris_s), 4), "m^2")
print("cotangent outer iterations      =", step + 1, " final step disp =", f"{disp:.2e}", "m")

uniform q = 1 surface area      = 38.2268 m^2
cotangent minimal surface area  = 38.2267 m^2
cotangent outer iterations      = 15  final step disp = 9.71e-04 m


# Minimal Catenoid Soap Film Between Two Coaxial Rings

Where uniform $q$ and cotangent weights differ most dramatically is a **catenoid soap film** stretched between two horizontal rings of radius $R = 3.0\text{ m}$ at $z = -1.6\text{ m}$ and $z = +1.6\text{ m}$. By Euler's theorem, the true minimal surface of revolution satisfies $r(z) = r_0 \cosh(z / r_0)$ with boundary condition $r_0 \cosh(1.6 / r_0) = 3.0\text{ m}$ ($r_0 \approx 2.388\text{ m}$). Uniform $q_k = 1$ depends on mesh aspect ratio and pinches the waist too far inward, whereas `Mite`'s cotangent iteration converges onto Euler's catenoid!

In [2]:
# solve r0 * cosh(1.6 / r0) = 3.0 by 15 Newton steps for exact Euler reference
r0_exact = 2.4
for _ in range(15):
    g_val = r0_exact * np.cosh(1.6 / r0_exact) - 3.0
    dg_val = np.cosh(1.6 / r0_exact) - (1.6 / r0_exact) * np.sinh(1.6 / r0_exact)
    r0_exact -= g_val / dg_val

# cylindrical triangle mesh: n_theta = 24 around, n_z = 9 rings (middle ring j = 4 sits at z = 0)
n_th, n_zr = 24, 9
R_ring, H_half = 3.0, 1.6
thetas = np.linspace(0.0, 2.0 * np.pi, n_th, endpoint=False)
z_levels = np.linspace(-H_half, H_half, n_zr)

xyz_cyl0 = np.zeros((n_th * n_zr, 3))
for j, zv in enumerate(z_levels):
    for i, th in enumerate(thetas):
        k = j * n_th + i
        xyz_cyl0[k] = [R_ring * np.cos(th), R_ring * np.sin(th), zv]

tris_cat = []
for j in range(n_zr - 1):
    for i in range(n_th):
        i_next = (i + 1) % n_th
        v00, v10 = j * n_th + i, j * n_th + i_next
        v01, v11 = (j + 1) * n_th + i, (j + 1) * n_th + i_next
        tris_cat.append((v00, v10, v11))
        tris_cat.append((v00, v11, v01))

edge_set_cat = set()
for a, b, c in tris_cat:
    for u, v in ((a, b), (b, c), (c, a)):
        edge_set_cat.add((min(u, v), max(u, v)))
edges_cat = sorted(edge_set_cat)
eidx_cat = {e: k for k, e in enumerate(edges_cat)}
C_cat = np.zeros((len(edges_cat), len(xyz_cyl0)))
for k, (a, b) in enumerate(edges_cat):
    C_cat[k, a], C_cat[k, b] = 1.0, -1.0

fixed_cat = list(range(0, n_th)) + list(range((n_zr - 1) * n_th, n_zr * n_th))
free_cat = [k for k in range(len(xyz_cyl0)) if k not in fixed_cat]
C_Nc, C_Fc = C_cat[:, free_cat], C_cat[:, fixed_cat]

# uniform q = 1 solve
xyz_cat_uni = xyz_cyl0.copy()
A_cu = C_Nc.T @ C_Nc
for axis in range(3):
    xyz_cat_uni[free_cat, axis] = np.linalg.solve(A_cu, -C_Nc.T @ C_Fc @ xyz_cat_uni[fixed_cat, axis])

# cotangent minimal surface solve
xyz_cat_min = xyz_cat_uni.copy()
for _ in range(20):
    q_c = cotangent_weights(xyz_cat_min, tris_cat, eidx_cat, len(edges_cat))
    Q_c = np.diag(q_c)
    A_cm = C_Nc.T @ Q_c @ C_Nc
    for axis in range(3):
        xyz_cat_min[free_cat, axis] = np.linalg.solve(A_cm, -C_Nc.T @ Q_c @ C_Fc @ xyz_cat_min[fixed_cat, axis])

waist_nodes = list(range((n_zr // 2) * n_th, (n_zr // 2 + 1) * n_th))
r_waist_uni = np.linalg.norm(xyz_cat_uni[waist_nodes, :2], axis=1).mean()
r_waist_min = np.linalg.norm(xyz_cat_min[waist_nodes, :2], axis=1).mean()

print("Euler exact catenoid neck r0   =", round(r0_exact, 4), "m")
print("uniform q = 1 waist radius     =", round(r_waist_uni, 4), "m  area =", round(mesh_area(xyz_cat_uni, tris_cat), 3), "m^2")
print("cotangent minimal waist radius =", round(r_waist_min, 4), "m  area =", round(mesh_area(xyz_cat_min, tris_cat), 3), "m^2")

# grey wireframe = uniform q, dark wireframe = true minimal catenoid
f_cat = q_c * np.linalg.norm(C_cat @ xyz_cat_min, axis=1)
colors_cat = [Color(0.60 - 0.60 * fv / f_cat.max(), 0.60 - 0.60 * fv / f_cat.max(), 0.60 - 0.60 * fv / f_cat.max(), 1.0) for fv in f_cat]

session_cat = Session("fdm_ext03_catenoid")
session_cat.add_network(Matrix.from_rows(C_cat.tolist()), Matrix.from_rows(xyz_cat_uni.tolist()), [GRAY] * len(edges_cat), [1.3] * len(edges_cat))
session_cat.add_network(Matrix.from_rows(C_cat.tolist()), Matrix.from_rows(xyz_cat_min.tolist()), colors_cat, [2.6] * len(edges_cat))
for net in range(2):
    for k in range(len(xyz_cyl0)):
        pt = session_cat.objects.points[net * len(xyz_cyl0) + k]
        pt.pointcolor = BLUE if k in fixed_cat else RED
        pt.width = 10.0 if k in fixed_cat else 5.5

show(session_cat, "fdm_ext03_catenoid")

Euler exact catenoid neck r0   = 2.4614 m
uniform q = 1 waist radius     = 1.8039 m  area = 58.929 m^2
cotangent minimal waist radius = 2.4499 m  area = 56.908 m^2


# Pneumatic Cushion Under Follower Normal Pressure

Now we implement `jax_fdm`'s local face load loop (`equilibrium/loads.py` with `is_load_local=True` inside `equilibrium_iterative_xyz`). Fix the entire $6\text{ m} \times 6\text{ m}$ perimeter frame at $z = 0\text{ m}$ and inflate the membrane with internal gauge pressure $\pi = 0.95\text{ kN/m}^{2}$. At every fixed point step:

$$
\vec{s}_t = \frac{1}{2}\left(x_{j} - x_{i}\right)\times\left(x_{k} - x_{i}\right), \qquad {\color{#2e9e4a} p_i(x)} = \sum_{t \ni i} \frac{1}{3}\,\pi\,\vec{s}_t
$$

Notice that as the pillow domes upward, ${\color{#2e9e4a} p_x(x)}$ and ${\color{#2e9e4a} p_y(x)}$ become non zero and push the sloping sides outward!

In [3]:
def follower_pressure_loads(xyz, tris, pressure):
    """Distribute local normal pressure pi * area_normal / 3 to triangle vertices."""
    loads = np.zeros_like(xyz)
    for i0, i1, i2 in tris:
        s_vec = 0.5 * np.cross(xyz[i1] - xyz[i0], xyz[i2] - xyz[i0])
        contrib = (pressure / 3.0) * s_vec
        loads[i0] += contrib
        loads[i1] += contrib
        loads[i2] += contrib
    return loads


xyz_pil = np.column_stack([xs.copy(), ys.copy(), np.zeros_like(xs)])
q_pil = np.full(len(edges_s), 1.8)
Q_pil = np.diag(q_pil)
A_pil = C_Ns.T @ Q_pil @ C_Ns
pressure_pi = 0.95

# compare 1 linear vertical load step vs converged follower normal load fixed point (tmax = 50)
p_vert = follower_pressure_loads(xyz_pil, tris_s, pressure_pi)
xyz_vert = xyz_pil.copy()
for axis in range(3):
    xyz_vert[free_s, axis] = np.linalg.solve(A_pil, p_vert[free_s, axis] - C_Ns.T @ Q_pil @ C_Fs @ xyz_vert[fixed_s, axis])

for it_p in range(50):
    p_fol = follower_pressure_loads(xyz_pil, tris_s, pressure_pi)

    # equation 6.32
    next_p = np.column_stack([
        np.linalg.solve(A_pil, p_fol[free_s, 0] - C_Ns.T @ Q_pil @ C_Fs @ xyz_pil[fixed_s, 0]),
        np.linalg.solve(A_pil, p_fol[free_s, 1] - C_Ns.T @ Q_pil @ C_Fs @ xyz_pil[fixed_s, 1]),
        np.linalg.solve(A_pil, p_fol[free_s, 2] - C_Ns.T @ Q_pil @ C_Fs @ xyz_pil[fixed_s, 2]),
    ])
    step_dp = np.linalg.norm(next_p - xyz_pil[free_s], axis=1).max()
    xyz_pil[free_s] = next_p
    if step_dp < 1e-9:
        break

f_pil = q_pil * np.linalg.norm(C_s @ xyz_pil, axis=1)
mid_s = len(xs) // 2
print("follower pressure iterations   =", it_p + 1, " step disp =", f"{step_dp:.2e}", "m")
print("vertical load apex z           =", round(xyz_vert[mid_s, 2], 3), "m")
print("follower normal load apex z    =", round(xyz_pil[mid_s, 2], 3), "m")
print("max lateral follower load p_xy =", round(np.linalg.norm(p_fol[free_s, :2], axis=1).max(), 3), "kN")

colors_pil = [Color(0.65 - 0.65 * fv / f_pil.max(), 0.65 - 0.65 * fv / f_pil.max(), 0.65 - 0.65 * fv / f_pil.max(), 1.0) for fv in f_pil]
session_pil = Session("fdm_ext03_pneumatic_pillow")
session_pil.add_network(Matrix.from_rows(C_s.tolist()), Matrix.from_rows(xyz_pil.tolist()), colors_pil, [2.6] * len(edges_s))
for k in range(len(xs)):
    pt = session_pil.objects.points[k]
    pt.pointcolor = BLUE if k in fixed_s else RED
    pt.width = 11.0 if k in fixed_s else 6.0

# draw green follower normal pressure vectors at interior nodes
for k in free_s[::2]:
    pv = p_fol[k] * 1.4
    arrow = Line(xyz_pil[k, 0], xyz_pil[k, 1], xyz_pil[k, 2], xyz_pil[k, 0] + pv[0], xyz_pil[k, 1] + pv[1], xyz_pil[k, 2] + pv[2])
    arrow.linecolor = GREEN
    arrow.width = 2.2
    arrow.arrowhead = Arrowhead.END
    session_pil.add_line(arrow)

show(session_pil, "fdm_ext03_pneumatic_pillow")

follower pressure iterations   = 15  step disp = 4.81e-10 m
vertical load apex z           = 0.702 m
follower normal load apex z    = 0.734 m
max lateral follower load p_xy = 0.256 kN


# Observation

**Freezing geometry dependent weights $Q(x)$ or follower loads $p(x)$ inside an outer fixed point loop turns a 4 line linear solver into a true surface mechanics engine.** Fifteen cotangent updates recover Euler's minimal catenoid neck to millimetre accuracy without thousands of explicit Euler steps (`Mite`), while updating triangle area normals inflates an ETFE cushion laterally as well as vertically (`jax_fdm`).